# Distribution-Free Uncertainty Quantification for Gastrointestinal Endoscopy Screening
## Conformal Prediction and Selective Classification for GERD and Polyp Triage

**Research Pipeline:**
1. Dataset audit & leakage-safe group split
2. Model training (ConvNeXt-Tiny, ViT-S/16)
3. Classification evaluation
4. Calibration analysis & temperature scaling
5. Split conformal prediction (LAC, APS, RAPS)
6. Selective classification / abstention
7. Journal-ready tables and figures
8. Output packaging

> **Leakage guarantee**: all augmented copies of the same primary image are kept in one split.


In [ ]:
# ── 1. Environment Setup ──────────────────────────────────────────────────
import subprocess, sys, os

def pip_install(pkg):
    try:
        __import__(pkg.split('[')[0].replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['timm', 'scikit-learn', 'pandas', 'matplotlib', 'tqdm']:
    pip_install(pkg)

import torch, platform, shutil
print(f'Python  : {platform.python_version()}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU mem : {mem:.1f} GB')
print(f'CWD     : {os.getcwd()}')

# ── Paths ─────────────────────────────────────────────────────────────────────
# TODO: replace <DATASET_NAME> with your Kaggle dataset slug
DATA_ROOT  = '/kaggle/input/<DATASET_NAME>'
OUTPUT_DIR = '/kaggle/working/gi_conformal_outputs'

for d in [
    OUTPUT_DIR,
    f'{OUTPUT_DIR}/tables',
    f'{OUTPUT_DIR}/figures',
    f'{OUTPUT_DIR}/checkpoints',
    f'{OUTPUT_DIR}/logs',
    f'{OUTPUT_DIR}/predictions',
]:
    os.makedirs(d, exist_ok=True)

print(f'\nDATA_ROOT  : {DATA_ROOT}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')


## 2. Quick / Full Run Switch
Set `QUICK_RUN = True` to smoke-test the pipeline end-to-end in minutes.

In [ ]:
# ── 2. Quick / Full Run Switch ────────────────────────────────────────────
QUICK_RUN = True   # <- set False for full paper-quality run

if QUICK_RUN:
    print('Quick QUICK_RUN = True  — small subset, 2 epochs, fast sanity check')
else:
    print('Full QUICK_RUN = False — full dataset, 20 epochs, paper-quality run')


## 3. Configuration

In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    IMAGE_SIZE: int = 224
    BATCH_SIZE: int = 16
    NUM_WORKERS: int = 2
    EPOCHS_QUICK: int = 2
    EPOCHS_FULL:  int = 20
    LR_HEAD:      float = 1e-3
    LR_BACKBONE:  float = 1e-5
    WEIGHT_DECAY: float = 1e-4
    EARLY_STOPPING_PATIENCE: int = 5
    GRAD_CLIP: float = 1.0
    BACKBONES: List[str] = field(default_factory=lambda: [
        'convnext_tiny',
        'vit_small_patch16_224',
    ])
    ALPHAS: List[float] = field(default_factory=lambda: [0.10, 0.05])
    TRAIN_FRAC: float = 0.60
    VAL_FRAC:   float = 0.10
    CAL_FRAC:   float = 0.15
    TEST_FRAC:  float = 0.15
    QUICK_MAX_PER_CLASS: int = 60
    SEED: int = 42
    NUM_CLASSES: int = 4

CFG = Config()
EPOCHS = CFG.EPOCHS_QUICK if QUICK_RUN else CFG.EPOCHS_FULL

import random, numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Epochs : {EPOCHS}')


## 4. Class Names & Mapping

In [ ]:
# ── 4. Class Names & Mapping (auto-detect) ────────────────────────────────
import re as _re, pathlib as _pl

def _norm(name: str) -> str:
    '''Normalize folder name: lowercase, strip spaces/hyphens/underscores.'''
    return _re.sub(r'[\s\-_]', '', name).lower()

# Canonical map: normalized_key -> (display_name, integer_label)
_CANONICAL = {
    'gerd':        ('GERD',         0),
    'gerdnormal':  ('GERD-Normal',  1),
    'polyp':       ('Polyp',        2),
    'polypnormal': ('Polyp-Normal', 3),
}

def auto_build_class_map(data_root: str) -> dict:
    root = _pl.Path(data_root)
    if not root.exists():
        print(f'WARNING: DATA_ROOT not found: {data_root}')
        return {}
    folders = sorted(p.name for p in root.iterdir() if p.is_dir())
    print(f'Discovered folders: {folders}')
    class_map, unmatched = {}, []
    for folder in folders:
        key = _norm(folder)
        if key in _CANONICAL:
            display, label = _CANONICAL[key]
            class_map[folder] = label
            print(f'  "{folder}" -> label={label} ({display})')
        else:
            unmatched.append(folder)
    if unmatched:
        print(f'\nWARNING: no canonical match for: {unmatched}')
        print('  Add entries to MANUAL_CLASS_MAP below if needed.')
    return class_map

# Manual override — add your own folder->label pairs here if auto-detect misses any
MANUAL_CLASS_MAP = {}  # example: {'MyGERDFolder': 0, 'MyPolypFolder': 2}

CLASS_MAP = auto_build_class_map(DATA_ROOT)
CLASS_MAP.update(MANUAL_CLASS_MAP)

if not CLASS_MAP:
    raise RuntimeError(
        'CLASS_MAP is empty — no folders matched canonical class names.\n'
        'Check that DATA_ROOT points to the correct directory and edit '
        'MANUAL_CLASS_MAP above to add the correct folder->label mapping.'
    )

# Build IDX_TO_CLASS with canonical display names
IDX_TO_CLASS = {}
for folder, label in CLASS_MAP.items():
    key = _norm(folder)
    display = _CANONICAL.get(key, (folder, label))[0]
    IDX_TO_CLASS[label] = display
for key, (display, label) in _CANONICAL.items():  # fill any gaps
    IDX_TO_CLASS.setdefault(label, display)

print(f'\nCLASS_MAP    : {CLASS_MAP}')
print(f'IDX_TO_CLASS : {IDX_TO_CLASS}')
print(f'NUM_CLASSES  : {len(IDX_TO_CLASS)}')


## 5. Dataset Scanning
Recursively scan `DATA_ROOT`, infer primary_id, build a master dataframe.

In [ ]:
# ── 5. Dataset Scanning ───────────────────────────────────────────────────
import re, pathlib
from PIL import Image
import pandas as pd

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

# ── Primary-ID inference ──────────────────────────────────────────────────────
_AUG_SUFFIXES = re.compile(
    r'(_aug\d*|_augmented\d*|_flip|_hflip|_vflip'
    r'|_rot\d*|_rotate\d*|_brightness\d*|_contrast\d*'
    r'|_crop\d*|_zoom\d*|_blur\d*|_noise\d*'
    r'|_gamma\d*|_saturation\d*|_hue\d*'
    r'|_elastic\d*|_warp\d*|_distort\d*'
    r'|_copy\d+|_v\d+|_\d+$)',
    re.IGNORECASE,
)

def infer_primary_id(stem: str) -> str:
    '''Strip augmentation suffixes to recover the original image stem.'''
    prev = None
    s = stem
    while s != prev:
        prev = s
        s = _AUG_SUFFIXES.sub('', s)
    return s.strip('_- ')

def scan_dataset(data_root: str, class_map: dict) -> pd.DataFrame:
    root = pathlib.Path(data_root)
    if not root.exists():
        raise FileNotFoundError(f'DATA_ROOT not found: {data_root}')

    discovered = sorted(p.name for p in root.iterdir() if p.is_dir())
    print(f'Discovered top-level folders: {discovered}')

    unknown = [f for f in discovered if f not in class_map]
    if unknown:
        print(f'\nUnknown folders (not in CLASS_MAP): {unknown}')
        print('   Edit CLASS_MAP in Cell 4 to add a mapping for each folder.')

    records = []
    for folder in root.iterdir():
        if not folder.is_dir() or folder.name not in class_map:
            continue
        label = class_map[folder.name]
        for img_path in folder.rglob('*'):
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            stem = img_path.stem
            pid  = infer_primary_id(stem)
            try:
                with Image.open(img_path) as im:
                    w, h = im.size
            except Exception:
                w, h = -1, -1
            records.append({
                'file_path' : str(img_path),
                'filename'  : img_path.name,
                'class_name': folder.name,
                'label'     : label,
                'primary_id': f'{folder.name}__{pid}',
                'width'     : w,
                'height'    : h,
            })

    df = pd.DataFrame(records)
    return df

df_all = scan_dataset(DATA_ROOT, CLASS_MAP)

# ── Primary-ID sanity check ──────────────────────────────────────────────────
n_images   = len(df_all)
n_pids     = df_all['primary_id'].nunique()
uniq_ratio = n_pids / max(n_images, 1)

print(f'\nTotal images   : {n_images}')
print(f'Unique PIDs    : {n_pids}')
print(f'Unique ratio   : {uniq_ratio:.3f}')

if uniq_ratio > 0.95:
    print('\nWARNING: almost every image has a unique primary_id.')
    print('   This means augmentation suffixes were NOT stripped successfully.')
    print('   Inspect filenames below and update _AUG_SUFFIXES regex in Cell 5.')
    print(df_all[['filename','primary_id']].head(20).to_string(index=False))
else:
    aug_ratio = n_images / max(n_pids, 1)
    print(f'Avg copies/primary: {aug_ratio:.1f}  OK')

print('\nSample filename -> primary_id:')
for _, row in df_all.sample(min(10, len(df_all)), random_state=CFG.SEED).iterrows():
    print(f'  {row["filename"]:50s} -> {row["primary_id"]}')

audit_path = f'{OUTPUT_DIR}/tables/dataset_audit.csv'
df_all.to_csv(audit_path, index=False)
print(f'\nAudit saved -> {audit_path}')
print(df_all.groupby('class_name')['filename'].count().rename('count').to_frame())


## 6. Dataset Visualizations

In [ ]:
# ── 6. Dataset Visualizations ─────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

FIG_DIR = f'{OUTPUT_DIR}/figures'
COLORS  = ['#4C72B0','#DD8452','#55A868','#C44E52']

def savefig(name):
    plt.savefig(f'{FIG_DIR}/{name}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{FIG_DIR}/{name}.pdf', bbox_inches='tight')
    plt.close()

# 6a. Class distribution
class_counts   = df_all.groupby('class_name').size().reindex(list(CLASS_MAP.keys()))
primary_counts = (df_all.drop_duplicates('primary_id')
                        .groupby('class_name').size()
                        .reindex(list(CLASS_MAP.keys())))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = range(len(CLASS_MAP))
axes[0].bar(x, class_counts.values, color=COLORS)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(list(CLASS_MAP.keys()), rotation=15, ha='right')
axes[0].set_title('Total Images per Class'); axes[0].set_ylabel('Count')

w = 0.35
axes[1].bar([i-w/2 for i in x], primary_counts.values, w, label='Primary', color=COLORS, alpha=0.9)
axes[1].bar([i+w/2 for i in x], class_counts.values,   w, label='Total (aug)', color=COLORS, alpha=0.5)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(list(CLASS_MAP.keys()), rotation=15, ha='right')
axes[1].set_title('Primary vs Augmented'); axes[1].set_ylabel('Count'); axes[1].legend()

plt.suptitle('Dataset Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout(); savefig('01_class_distribution')
print('Saved: 01_class_distribution')

# 6b. Image size distribution
df_valid = df_all[df_all['width'] > 0]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(df_valid['width'],  bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Width (px)');  axes[0].set_title('Image Width Distribution')
axes[1].hist(df_valid['height'], bins=30, color='salmon',    edgecolor='white')
axes[1].set_xlabel('Height (px)'); axes[1].set_title('Image Height Distribution')
plt.tight_layout(); savefig('02_image_size_distribution')
print('Saved: 02_image_size_distribution')

# 6c. Random image grid per class
n_per_class = 4
classes = list(CLASS_MAP.keys())
fig, axes = plt.subplots(len(classes), n_per_class, figsize=(n_per_class*3, len(classes)*3))
for r, cls in enumerate(classes):
    cls_df  = df_all[df_all['class_name'] == cls]
    samples = cls_df.sample(min(n_per_class, len(cls_df)), random_state=CFG.SEED)
    for c, (_, row) in enumerate(samples.iterrows()):
        ax = axes[r][c]
        try:
            img = Image.open(row['file_path']).convert('RGB'); ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
        ax.axis('off')
        if c == 0:
            ax.set_ylabel(cls, fontsize=9)
plt.suptitle('Random Sample Grid per Class', fontsize=13, fontweight='bold')
plt.tight_layout(); savefig('03_image_grid_per_class')
print('Saved: 03_image_grid_per_class')

# 6d. Augmented copies from same primary_id
pid_counts = df_all.groupby('primary_id').size().sort_values(ascending=False)
if len(pid_counts) > 0:
    best_pid  = pid_counts.index[0]
    aug_group = df_all[df_all['primary_id'] == best_pid]
    n_show    = min(6, len(aug_group))
    fig, axes_g = plt.subplots(1, n_show, figsize=(n_show*3, 3))
    if n_show == 1: axes_g = [axes_g]
    for i, (_, row) in enumerate(aug_group.head(n_show).iterrows()):
        try:
            img = Image.open(row['file_path']).convert('RGB'); axes_g[i].imshow(img)
        except Exception:
            axes_g[i].text(0.5, 0.5, 'N/A', ha='center', va='center')
        axes_g[i].set_title(row['filename'][:18], fontsize=7); axes_g[i].axis('off')
    plt.suptitle(f'Augmented copies — primary_id: {best_pid}', fontsize=10)
    plt.tight_layout(); savefig('04_augmented_copies_example')
    print('Saved: 04_augmented_copies_example')


## 7. Leakage-Safe Group Split

All augmented copies of the same primary image are guaranteed to stay in one split.
We use `StratifiedGroupKFold` on primary images, then propagate split labels to
all augmented copies. Hard assert checks stop execution if leakage is found.


In [ ]:
# ── 7. Leakage-Safe Group Split ──────────────────────────────────────────
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit
from collections import Counter

def _quick_subsample(df, max_per_class, seed):
    '''Subsample per class without pandas FutureWarning.'''
    parts = []
    for _, grp in df.groupby('class_name'):
        parts.append(grp.sample(min(max_per_class, len(grp)), random_state=seed))
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[:0]

def _safe_group_split(pids, labels, n_splits_target, seed):
    '''Try StratifiedGroupKFold; fall back gracefully on any constraint violation.'''
    n         = len(pids)
    n_unique  = len(set(pids))
    min_cls   = min(Counter(labels).values()) if len(labels) > 0 else 1

    # SGKF requires n_splits <= n_unique AND n_splits <= min_class_count
    n_splits = min(n_splits_target, n_unique, min_cls)
    n_splits = max(2, n_splits)
    if n_splits < n_splits_target:
        print(f'    Note: n_splits {n_splits_target}->{n_splits} (n={n}, min_cls={min_cls})')

    # Attempt 1: StratifiedGroupKFold
    try:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        tr, te = next(iter(sgkf.split(pids, labels, pids)))
        return list(tr), list(te)
    except Exception:
        pass

    # Attempt 2: StratifiedShuffleSplit (no per-fold-per-class constraint)
    test_size = min(0.49, 1.0 / n_splits_target)
    print(f'    Fallback: StratifiedShuffleSplit(test_size={test_size:.2f})')
    try:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        tr, te = next(iter(sss.split(pids, labels)))
        return list(tr), list(te)
    except Exception:
        pass

    # Attempt 3: deterministic index split (last resort)
    n_test = max(1, min(n - 1, int(round(n * test_size))))
    print(f'    Last-resort index split: {n - n_test} train / {n_test} test')
    return list(range(n - n_test)), list(range(n - n_test, n))


def make_splits(df: pd.DataFrame, cfg, quick_run: bool, seed: int):
    df_prim = df.drop_duplicates('primary_id').reset_index(drop=True)

    if len(df_prim) == 0:
        raise ValueError('No primary images found! Check CLASS_MAP in Cell 4.')

    _per_image_mode = False   # track whether we bypass group-aware split

    if quick_run:
        df_prim_sub = _quick_subsample(df_prim, cfg.QUICK_MAX_PER_CLASS, seed)
        n_pids = len(df_prim_sub)
        n_cls  = max(1, df_prim_sub['class_name'].nunique())
        print(f'Quick mode: {n_pids} primary IDs across {n_cls} classes')

        if n_pids < n_cls * 4:
            # Too few unique primary IDs to do group-aware splitting.
            # This happens when all augmented copies share the same filename stem
            # (e.g. 'GERD_1.jpg' ... 'GERD_5844.jpg' all strip to 'GERD').
            # Fall back: sample raw images and give each a synthetic unique PID.
            print(f'  Only ~{n_pids // n_cls} unique primary ID per class detected.')
            print('  The augmentation suffix regex may not match your filename pattern.')
            print('  Quick-mode FALLBACK: per-image split (group-leakage check bypassed).')
            print('  Edit _AUG_SUFFIXES in Cell 5 to fix for full-mode runs.\n')
            n_img = max(cfg.QUICK_MAX_PER_CLASS, 40)  # raw images per class
            df_prim = _quick_subsample(df, n_img, seed).copy().reset_index(drop=True)
            # Give each sampled image a unique synthetic PID so splits are disjoint
            df_prim['primary_id'] = [f'__img_{i}__' for i in range(len(df_prim))]
            _per_image_mode = True
        else:
            df_prim = df_prim_sub

    if len(df_prim) < 4:
        raise ValueError(
            f'Only {len(df_prim)} samples available for splitting. '
            'Increase QUICK_MAX_PER_CLASS or check DATA_ROOT.'
        )

    pids   = df_prim['primary_id'].values
    labels = df_prim['label'].values

    # Step 1: carve off test + cal (~30%)
    test_cal_frac = cfg.TEST_FRAC + cfg.CAL_FRAC
    train_val_idx, test_cal_idx = _safe_group_split(
        pids, labels, max(2, round(1 / test_cal_frac)), seed)
    df_trainval = df_prim.iloc[train_val_idx].reset_index(drop=True)
    df_testcal  = df_prim.iloc[test_cal_idx].reset_index(drop=True)

    # Step 2: split test_cal -> cal / test
    if len(df_testcal) < 2:
        df_cal  = df_testcal.iloc[:0].copy()
        df_test = df_testcal.copy()
    else:
        cal_frac_of_tc = cfg.CAL_FRAC / test_cal_frac
        cal_idx, test_idx = _safe_group_split(
            df_testcal['primary_id'].values, df_testcal['label'].values,
            max(2, round(1 / cal_frac_of_tc)), seed + 1)
        df_cal  = df_testcal.iloc[cal_idx].reset_index(drop=True)
        df_test = df_testcal.iloc[test_idx].reset_index(drop=True)

    # Step 3: split train_val -> train / val
    if len(df_trainval) < 2:
        df_train = df_trainval.copy()
        df_val   = df_trainval.iloc[:0].copy()
    else:
        val_frac_of_tv = cfg.VAL_FRAC / (cfg.TRAIN_FRAC + cfg.VAL_FRAC)
        train_idx, val_idx = _safe_group_split(
            df_trainval['primary_id'].values, df_trainval['label'].values,
            max(2, round(1 / val_frac_of_tv)), seed + 2)
        df_train = df_trainval.iloc[train_idx].reset_index(drop=True)
        df_val   = df_trainval.iloc[val_idx].reset_index(drop=True)

    return df_train, df_val, df_cal, df_test, _per_image_mode


df_prim_train, df_prim_val, df_prim_cal, df_prim_test, _per_image_mode = make_splits(
    df_all, CFG, QUICK_RUN, CFG.SEED)

# Expand primary splits to include all augmented copies (unless per-image fallback)
def expand_to_augmented(df_all, df_prim_split):
    pids = set(df_prim_split['primary_id'])
    return df_all[df_all['primary_id'].isin(pids)].reset_index(drop=True)

if _per_image_mode:
    # Per-image fallback: splits ARE the final dataframes (no augmented expansion)
    df_train = df_prim_train.copy()
    df_val   = df_prim_val.copy()
    df_cal   = df_prim_cal.copy()
    df_test  = df_prim_test.copy()
else:
    if QUICK_RUN:
        all_pids = (set(df_prim_train['primary_id']) | set(df_prim_val['primary_id'])
                  | set(df_prim_cal['primary_id'])   | set(df_prim_test['primary_id']))
        df_use = df_all[df_all['primary_id'].isin(all_pids)]
    else:
        df_use = df_all
    df_train = expand_to_augmented(df_use, df_prim_train)
    df_val   = expand_to_augmented(df_use, df_prim_val)
    df_cal   = expand_to_augmented(df_use, df_prim_cal)
    df_test  = expand_to_augmented(df_use, df_prim_test)

# ── Hard leakage checks ──────────────────────────────────────────────────────
split_sets = {
    'train': set(df_train['primary_id']),
    'val'  : set(df_val['primary_id']),
    'cal'  : set(df_cal['primary_id']),
    'test' : set(df_test['primary_id']),
}
leak_found = False
for a, b in [('train','val'),('train','cal'),('train','test'),
             ('val','cal'),  ('val','test'), ('cal','test')]:
    overlap = split_sets[a] & split_sets[b]
    if overlap:
        print(f'LEAKAGE between {a} and {b}: {len(overlap)} PIDs!')
        leak_found = True
if leak_found:
    raise RuntimeError('Data leakage detected — check split logic.')
else:
    print('No data leakage detected across all split pairs.')

for name, sdf in [('train',df_train),('val',df_val),('cal',df_cal),('test',df_test)]:
    counts = sdf.groupby('class_name').size() if len(sdf) > 0 else pd.Series(dtype=int)
    print(f'  {name:6s}: {len(sdf):5d} images | '
          + ' | '.join(f'{c}:{n}' for c,n in counts.items()))

df_split = pd.concat([
    df_train.assign(split='train'), df_val.assign(split='val'),
    df_cal.assign(split='cal'),     df_test.assign(split='test'),
], ignore_index=True)
df_split.to_csv(f'{OUTPUT_DIR}/tables/split_file.csv', index=False)
print(f'Split file saved -> {OUTPUT_DIR}/tables/split_file.csv')


## 8. Dataloaders

In [ ]:
# ── 8. Dataloaders ────────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((CFG.IMAGE_SIZE + 32, CFG.IMAGE_SIZE + 32)),
    T.RandomCrop(CFG.IMAGE_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    T.CenterCrop(CFG.IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class GIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['file_path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (CFG.IMAGE_SIZE, CFG.IMAGE_SIZE))
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])

def make_loader(df, transform, shuffle=False, batch_size=None):
    bs = batch_size or CFG.BATCH_SIZE
    ds = GIDataset(df, transform)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      num_workers=CFG.NUM_WORKERS, pin_memory=True,
                      drop_last=shuffle)

train_loader = make_loader(df_train, train_transform, shuffle=True)
val_loader   = make_loader(df_val,   eval_transform)
cal_loader   = make_loader(df_cal,   eval_transform)
test_loader  = make_loader(df_test,  eval_transform)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Cal   batches : {len(cal_loader)}')
print(f'Test  batches : {len(test_loader)}')


## 9. Model Definitions
ConvNeXt-Tiny and ViT-S/16 from `timm` with 4-class head. Falls back to random init if pretrained weights cannot download.

In [ ]:
# ── 9. Models ─────────────────────────────────────────────────────────────
import timm
import torch.nn as nn

def build_model(backbone_name: str, num_classes: int = 4,
                pretrained: bool = True) -> nn.Module:
    try:
        model = timm.create_model(backbone_name, pretrained=pretrained,
                                  num_classes=num_classes)
        print(f'  {backbone_name} loaded (pretrained={pretrained})')
    except Exception as e:
        print(f'  pretrained=True failed ({e}). Trying random init...')
        model = timm.create_model(backbone_name, pretrained=False,
                                  num_classes=num_classes)
        print(f'  {backbone_name} loaded (pretrained=False — random init)')
    return model

def get_param_groups(model: nn.Module, lr_head: float, lr_backbone: float,
                     weight_decay: float):
    head_params, backbone_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(k in name for k in ('head', 'classifier', 'fc')):
            head_params.append(p)
        else:
            backbone_params.append(p)
    return [
        {'params': head_params,     'lr': lr_head,     'weight_decay': weight_decay},
        {'params': backbone_params, 'lr': lr_backbone, 'weight_decay': weight_decay},
    ]

print('Building models:')
for bb in CFG.BACKBONES:
    m = build_model(bb, CFG.NUM_CLASSES)
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'    Params: {n_params:.1f}M')
    del m


## 10. Training
AdamW with separate LRs for backbone/head, mixed precision, cosine LR, early stopping, and auto-OOM batch-size halving.

In [ ]:
# ── 10. Training ──────────────────────────────────────────────────────────
import time, copy
from torch.amp import GradScaler, autocast

_AMP_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_one_epoch(model, loader, optimizer, criterion, scaler, device, grad_clip):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=_AMP_DEVICE, enabled=(device.type == 'cuda')):
            out  = model(imgs)
            loss = criterion(out, labels)
        if device.type == 'cuda':
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        bs = labels.size(0)
        total_loss += loss.item() * bs
        correct    += (out.argmax(1) == labels).sum().item()
        total      += bs
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast(device_type=_AMP_DEVICE, enabled=(device.type == 'cuda')):
            out  = model(imgs)
            loss = criterion(out, labels)
        bs = labels.size(0)
        total_loss += loss.item() * bs
        correct    += (out.argmax(1) == labels).sum().item()
        total      += bs
    return total_loss / total, correct / total

def train_model(backbone_name, cfg, device, tr_loader, va_loader, epochs, output_dir):
    sep = '='*60
    print('\n' + sep)
    print(f'Training: {backbone_name}')
    print(sep)

    batch_size = cfg.BATCH_SIZE
    current_tr_loader = tr_loader
    while True:
        try:
            model     = build_model(backbone_name, cfg.NUM_CLASSES).to(device)
            optimizer = torch.optim.AdamW(
                get_param_groups(model, cfg.LR_HEAD, cfg.LR_BACKBONE, cfg.WEIGHT_DECAY))
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=epochs)
            criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
            scaler    = GradScaler(device=_AMP_DEVICE,
                                   enabled=(device.type == 'cuda'))
            # OOM probe
            imgs, lab = next(iter(current_tr_loader))
            imgs, lab = imgs.to(device), lab.to(device)
            with autocast(device_type=_AMP_DEVICE, enabled=(device.type=='cuda')):
                loss = criterion(model(imgs), lab)
            if device.type == 'cuda':
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            batch_size = max(4, batch_size // 2)
            print(f'  OOM! Reducing batch to {batch_size}')
            current_tr_loader = make_loader(df_train, train_transform,
                                            shuffle=True, batch_size=batch_size)

    log_rows, best_val_acc, patience_cnt = [], 0.0, 0
    ckpt = f'{output_dir}/checkpoints/{backbone_name.replace("/","_")}_best.pt'

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, current_tr_loader, optimizer,
                                          criterion, scaler, device, cfg.GRAD_CLIP)
        va_loss, va_acc = evaluate(model, va_loader, criterion, device)
        scheduler.step()
        elapsed = time.time() - t0
        print(f'  Ep {epoch:02d}/{epochs} | '
              f'tr {tr_loss:.4f}/{tr_acc:.4f} | '
              f'va {va_loss:.4f}/{va_acc:.4f} | {elapsed:.1f}s')
        log_rows.append({'epoch':epoch,'tr_loss':tr_loss,'tr_acc':tr_acc,
                         'va_loss':va_loss,'va_acc':va_acc})
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), ckpt)
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= cfg.EARLY_STOPPING_PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    log_df = pd.DataFrame(log_rows)
    log_df.to_csv(f'{output_dir}/logs/{backbone_name.replace("/","_")}_training.csv',
                  index=False)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(log_df['epoch'], log_df['tr_loss'], label='Train')
    axes[0].plot(log_df['epoch'], log_df['va_loss'], label='Val')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{backbone_name} Loss'); axes[0].legend()
    axes[1].plot(log_df['epoch'], log_df['tr_acc'], label='Train')
    axes[1].plot(log_df['epoch'], log_df['va_acc'], label='Val')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'{backbone_name} Accuracy'); axes[1].legend()
    plt.tight_layout()
    savefig(f'05_{backbone_name.replace("/","_")}_training_curves')

    print(f'  Best val acc: {best_val_acc:.4f}')
    return model, log_df

trained_models = {}
for backbone in CFG.BACKBONES:
    m, log = train_model(backbone, CFG, DEVICE, train_loader, val_loader,
                         EPOCHS, OUTPUT_DIR)
    trained_models[backbone] = m


## 11. Test Set Evaluation

In [ ]:
# ── 11. Test Evaluation ───────────────────────────────────────────────────
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score, accuracy_score)

@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_logits, all_probs, all_labels = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with autocast(device_type=_AMP_DEVICE, enabled=(device.type=='cuda')):
            logits = model(imgs)
        probs = torch.softmax(logits, dim=-1)
        all_logits.append(logits.cpu())
        all_probs.append(probs.cpu())
        all_labels.append(labels)
    return (torch.cat(all_logits).numpy(),
            torch.cat(all_probs).numpy(),
            torch.cat(all_labels).numpy())

def evaluate_model(backbone_name, model, test_loader, device, output_dir):
    logits, probs, labels = get_predictions(model, test_loader, device)
    preds = probs.argmax(axis=1)

    acc    = accuracy_score(labels, preds)
    mac_f1 = f1_score(labels, preds, average='macro',    zero_division=0)
    wgt_f1 = f1_score(labels, preds, average='weighted', zero_division=0)
    try:
        auroc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except Exception:
        auroc = float('nan')

    print(f'\n── {backbone_name} ──────────────────────────')
    print(f'  Accuracy   : {acc:.4f}')
    print(f'  Macro-F1   : {mac_f1:.4f}')
    print(f'  Weighted-F1: {wgt_f1:.4f}')
    print(f'  AUROC(OVR) : {auroc:.4f}')

    report    = classification_report(labels, preds,
                                      target_names=list(IDX_TO_CLASS.values()),
                                      output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).T
    report_df.to_csv(
        f'{output_dir}/tables/{backbone_name.replace("/","_")}_classification_report.csv')

    # Confusion matrix
    cm  = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ticks = list(IDX_TO_CLASS.values())
    ax.set_xticks(range(CFG.NUM_CLASSES)); ax.set_xticklabels(ticks, rotation=30, ha='right')
    ax.set_yticks(range(CFG.NUM_CLASSES)); ax.set_yticklabels(ticks)
    for i in range(CFG.NUM_CLASSES):
        for j in range(CFG.NUM_CLASSES):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — {backbone_name}')
    plt.tight_layout()
    savefig(f'06_{backbone_name.replace("/","_")}_confusion_matrix')

    # ── Save human-readable CSV predictions ─────────────────────────────────
    entropy_  = -(probs * np.log(probs + 1e-10)).sum(axis=1)
    pred_df   = pd.DataFrame({
        'file_path'  : df_test['file_path'].values,
        'filename'   : df_test['filename'].values,
        'class_name' : df_test['class_name'].values,
        'true_label' : labels,
        'true_class' : [IDX_TO_CLASS[l] for l in labels],
        'pred_label' : preds,
        'pred_class' : [IDX_TO_CLASS[p] for p in preds],
        'correct'    : (preds == labels).astype(int),
        'confidence' : probs.max(axis=1),
        'confidence_ts': test_results[backbone_name]['probs_ts'].max(axis=1)
                          if 'probs_ts' in test_results.get(backbone_name, {}) else float('nan'),
        'entropy'    : entropy_,
    })
    for c in range(CFG.NUM_CLASSES):
        cn = IDX_TO_CLASS[c]
        pred_df[f'prob_{cn}']    = probs[:, c]
        pred_df[f'prob_ts_{cn}'] = test_results[backbone_name]['probs_ts'][:, c] \
                                    if 'probs_ts' in test_results.get(backbone_name, {}) \
                                    else float('nan')
    csv_pred_path = f'{output_dir}/predictions/{backbone_name.replace("/","_")}_test_predictions.csv'
    pred_df.to_csv(csv_pred_path, index=False)
    print(f'  CSV predictions saved -> {csv_pred_path}')

    np.savez(f'{output_dir}/predictions/{backbone_name.replace("/","_")}_test.npz',
             logits=logits, probs=probs, labels=labels)

    return {'backbone':backbone_name,'accuracy':acc,'macro_f1':mac_f1,
            'weighted_f1':wgt_f1,'auroc':auroc,'logits':logits,
            'probs':probs,'labels':labels}

test_results = {}
for backbone, model in trained_models.items():
    test_results[backbone] = evaluate_model(backbone, model, test_loader,
                                            DEVICE, OUTPUT_DIR)


## 12. Calibration Analysis & Temperature Scaling

Temperature scaling is fitted on the **calibration split** exclusively. Evaluation is then performed on the **test split** to avoid leakage.


In [ ]:
# ── 12. Calibration ───────────────────────────────────────────────────────
from scipy.optimize import minimize_scalar

def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    '''Expected and Maximum Calibration Error.'''
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    bin_edges   = np.linspace(0, 1, n_bins + 1)
    ece, mce    = 0.0, 0.0
    total       = len(labels)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            continue
        acc  = (predictions[mask] == labels[mask]).mean()
        conf = confidences[mask].mean()
        frac = mask.sum() / total
        ece += frac * abs(acc - conf)
        mce  = max(mce, abs(acc - conf))
    return ece, mce

def reliability_diagram(probs, labels, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    bin_edges   = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_fracs = [], [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            bin_accs.append(0); bin_confs.append((lo+hi)/2); bin_fracs.append(0)
        else:
            bin_accs.append((predictions[mask] == labels[mask]).mean())
            bin_confs.append(confidences[mask].mean())
            bin_fracs.append(mask.sum() / len(labels))
    return np.array(bin_accs), np.array(bin_confs), np.array(bin_fracs)

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.0)

    def forward(self, logits):
        return logits / self.temperature

    def fit(self, logits_np: np.ndarray, labels_np: np.ndarray):
        '''Optimize temperature on calibration data by minimizing NLL.'''
        logits_t = torch.tensor(logits_np, dtype=torch.float32)
        labels_t = torch.tensor(labels_np, dtype=torch.long)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS([self.temperature], max_iter=500, lr=0.01,
                                       line_search_fn='strong_wolfe')
        def closure():
            optimizer.zero_grad()
            loss = criterion(logits_t / self.temperature, labels_t)
            loss.backward()
            return loss
        optimizer.step(closure)
        self.temperature.data.clamp_(0.1, 10.0)
        return self

def calibrate_model(backbone_name, model, cal_loader, test_res, device, output_dir):
    cal_logits, cal_probs, cal_labels = get_predictions(model, cal_loader, device)
    test_logits = test_res['logits']
    test_labels = test_res['labels']

    ece_pre, mce_pre = compute_ece(cal_probs, cal_labels)
    print(f'\n── {backbone_name} calibration ──────────────────')
    print(f'  ECE (pre, cal set)  : {ece_pre:.4f}')
    print(f'  MCE (pre, cal set)  : {mce_pre:.4f}')

    ts = TemperatureScaler()
    ts.fit(cal_logits, cal_labels)
    T  = ts.temperature.item()
    print(f'  T*                  : {T:.4f}')

    test_logits_t  = torch.tensor(test_logits, dtype=torch.float32)
    test_probs_ts  = torch.softmax(test_logits_t / T, dim=-1).numpy()
    ece_post, mce_post = compute_ece(test_probs_ts, test_labels)
    print(f'  ECE (post, test)    : {ece_post:.4f}')
    print(f'  MCE (post, test)    : {mce_post:.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, prbs, lbl, tag in [
        (axes[0], cal_probs,    cal_labels,  'Pre-TS (Cal)'),
        (axes[1], test_probs_ts, test_labels, f'Post-TS T={T:.2f} (Test)'),
    ]:
        b_accs, b_confs, _ = reliability_diagram(prbs, lbl)
        ax.bar(b_confs, b_accs, width=1/15, alpha=0.7, label='Accuracy')
        ax.plot([0,1],[0,1],'k--', lw=1, label='Perfect')
        ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
        ax.set_title(tag); ax.legend(fontsize=8)
        ece_v, _ = compute_ece(prbs, lbl)
        ax.text(0.05, 0.9, f'ECE={ece_v:.3f}', transform=ax.transAxes)
    plt.suptitle(f'Reliability Diagram — {backbone_name}', fontweight='bold')
    plt.tight_layout()
    savefig(f'07_{backbone_name.replace("/","_")}_reliability')

    return {'backbone':backbone_name,'T':T,
            'ece_pre':ece_pre,'mce_pre':mce_pre,
            'ece_post':ece_post,'mce_post':mce_post,
            'test_probs_ts':test_probs_ts,
            'cal_logits':cal_logits,'cal_labels':cal_labels}

cal_results = {}
for backbone, model in trained_models.items():
    cr = calibrate_model(backbone, model, cal_loader,
                         test_results[backbone], DEVICE, OUTPUT_DIR)
    cal_results[backbone] = cr
    test_results[backbone]['probs_ts'] = cr['test_probs_ts']


## 13. Split Conformal Prediction

Three methods:
- **LAC**: score = 1 – p̂_y
- **APS**: cumulative softmax of sorted classes up to true class
- **RAPS**: APS + regularisation penalty for large sets

Calibration uses the calibration split; coverage is evaluated on the test split.


In [ ]:
# ── 13. Conformal Prediction ──────────────────────────────────────────────

def lac_scores(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
    '''LAC nonconformity score: 1 minus softmax of true class.'''
    return 1.0 - probs[np.arange(len(labels)), labels]

def aps_scores(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
    '''APS: cumulative sum up to and including the true class.'''
    order  = np.argsort(-probs, axis=1)
    scores = np.zeros(len(labels))
    for i, (row_probs, lbl) in enumerate(zip(probs, labels)):
        cum = 0.0
        for k in order[i]:
            cum += row_probs[k]
            if k == lbl:
                scores[i] = cum; break
    return scores

def raps_scores(probs: np.ndarray, labels: np.ndarray,
                lam: float = 0.01, k_reg: int = 1) -> np.ndarray:
    '''RAPS: APS plus regularisation penalty for rank beyond k_reg.'''
    order  = np.argsort(-probs, axis=1)
    scores = np.zeros(len(labels))
    for i, (row_probs, lbl) in enumerate(zip(probs, labels)):
        cum = 0.0
        for rank_k, k in enumerate(order[i]):
            cum += row_probs[k] + lam * max(0, rank_k + 1 - k_reg)
            if k == lbl:
                scores[i] = cum; break
    return scores

def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    level = np.ceil((n + 1) * (1 - alpha)) / n
    return float(np.quantile(scores, min(level, 1.0)))

def predict_sets_lac(probs: np.ndarray, qhat: float) -> list:
    return [[k for k in range(probs.shape[1]) if 1.0 - probs[i,k] <= qhat]
            for i in range(len(probs))]

def predict_sets_aps(probs: np.ndarray, qhat: float) -> list:
    order = np.argsort(-probs, axis=1)
    sets  = []
    for i, row_probs in enumerate(probs):
        cum, pred_set = 0.0, []
        for k in order[i]:
            cum += row_probs[k]; pred_set.append(k)
            if cum >= qhat: break
        sets.append(pred_set)
    return sets

def predict_sets_raps(probs: np.ndarray, qhat: float,
                       lam: float = 0.01, k_reg: int = 1) -> list:
    order = np.argsort(-probs, axis=1)
    sets  = []
    for i, row_probs in enumerate(probs):
        cum, pred_set = 0.0, []
        for rank_k, k in enumerate(order[i]):
            cum += row_probs[k] + lam * max(0, rank_k + 1 - k_reg)
            pred_set.append(k)
            if cum >= qhat: break
        sets.append(pred_set)
    return sets

def eval_conformal(pred_sets: list, labels: np.ndarray, num_classes: int):
    n = len(labels)
    coverage = sum(lbl in s for lbl, s in zip(labels, pred_sets)) / n
    avg_sz   = sum(len(s) for s in pred_sets) / n
    sing_r   = sum(1 for s in pred_sets if len(s) == 1) / n
    cls_cov = {}; cls_sz = {}; cls_sing = {}
    for c in range(num_classes):
        mask   = (labels == c)
        sets_c = [s for s, m in zip(pred_sets, mask) if m]
        if len(sets_c) == 0:
            cls_cov[c] = cls_sz[c] = cls_sing[c] = float('nan'); continue
        cls_cov[c]  = sum(c in s for s in sets_c) / len(sets_c)
        cls_sz[c]   = sum(len(s) for s in sets_c) / len(sets_c)
        cls_sing[c] = sum(1 for s in sets_c if len(s)==1) / len(sets_c)
    return coverage, avg_sz, sing_r, cls_cov, cls_sz, cls_sing

conformal_rows  = []
CONFORMAL_SETS  = {}   # {backbone: {method: {alpha: pred_sets}}}

for backbone in CFG.BACKBONES:
    CONFORMAL_SETS[backbone] = {}
    T           = cal_results[backbone]['T']
    cal_log     = cal_results[backbone]['cal_logits']
    cal_lbl_arr = cal_results[backbone]['cal_labels']
    cal_probs_ts = torch.softmax(
        torch.tensor(cal_log) / T, dim=-1).numpy()

    test_probs_ts  = test_results[backbone]['probs_ts']
    test_labels_   = test_results[backbone]['labels']

    for method in ['LAC', 'APS', 'RAPS']:
        CONFORMAL_SETS[backbone][method] = {}
        for alpha in CFG.ALPHAS:
            if method == 'LAC':
                cal_s = lac_scores(cal_probs_ts, cal_lbl_arr)
            elif method == 'APS':
                cal_s = aps_scores(cal_probs_ts, cal_lbl_arr)
            else:
                cal_s = raps_scores(cal_probs_ts, cal_lbl_arr)

            qhat = conformal_quantile(cal_s, alpha)

            if method == 'LAC':
                pred_sets = predict_sets_lac(test_probs_ts, qhat)
            elif method == 'APS':
                pred_sets = predict_sets_aps(test_probs_ts, qhat)
            else:
                pred_sets = predict_sets_raps(test_probs_ts, qhat)

            CONFORMAL_SETS[backbone][method][alpha] = pred_sets

            cov, avg_sz, sing_r, cls_cov, cls_sz, cls_sing = \
                eval_conformal(pred_sets, test_labels_, CFG.NUM_CLASSES)

            row = {'model':backbone,'method':method,'alpha':alpha,
                   'target_coverage':1-alpha,'empirical_coverage':cov,
                   'avg_set_size':avg_sz,'singleton_rate':sing_r,'qhat':qhat}
            for c in range(CFG.NUM_CLASSES):
                cn = IDX_TO_CLASS[c]
                row[f'cov_{cn}']  = cls_cov[c]
                row[f'sz_{cn}']   = cls_sz[c]
                row[f'sing_{cn}'] = cls_sing[c]
            conformal_rows.append(row)

            print(f'  {backbone[:18]:18s} | {method} | alpha={alpha:.2f} | '
                  f'target={1-alpha:.2f} emp={cov:.3f} sz={avg_sz:.2f} sing={sing_r:.2f}')

conformal_df = pd.DataFrame(conformal_rows)
conformal_df.to_csv(f'{OUTPUT_DIR}/tables/table7_conformal_coverage.csv', index=False)
print('\nConformal table saved.')


## 14. Selective Classification / Abstention

In [ ]:
# ── 14. Selective Classification ──────────────────────────────────────────

def aurc(risks: np.ndarray, coverages: np.ndarray) -> float:
    '''Area Under Risk-Coverage curve (trapezoidal).'''
    idx  = np.argsort(-coverages)
    return float(np.trapz(risks[idx], coverages[idx]))

def selective_metrics(scores, preds, labels, n_thresh=100):
    thresholds = np.linspace(scores.min(), scores.max(), n_thresh)
    covs, accs, risks = [], [], []
    for thr in thresholds:
        mask = scores >= thr
        if mask.sum() == 0: continue
        cov  = mask.mean()
        acc  = (preds[mask] == labels[mask]).mean()
        covs.append(cov); accs.append(acc); risks.append(1.0 - acc)
    covs  = np.array(covs)
    accs  = np.array(accs)
    risks = np.array(risks)

    def at_cov(target):
        if len(covs) == 0: return float('nan'), float('nan')
        idx = np.argmin(np.abs(covs - target))
        return accs[idx], risks[idx]

    a100, _ = at_cov(1.0)
    a90,  _ = at_cov(0.90)
    a80,  _ = at_cov(0.80)
    arc     = aurc(risks, covs) if len(risks) > 1 else float('nan')
    return covs, accs, risks, a100, a90, a80, arc

selective_rows = []
SELECTIVE_RES  = {}

for backbone in CFG.BACKBONES:
    SELECTIVE_RES[backbone] = {}
    test_probs_raw = test_results[backbone]['probs']
    test_probs_ts  = test_results[backbone]['probs_ts']
    test_labels_   = test_results[backbone]['labels']
    preds_raw      = test_probs_raw.argmax(axis=1)
    preds_ts       = test_probs_ts.argmax(axis=1)

    score_cfgs = [
        ('MSP_raw', test_probs_raw.max(axis=1), preds_raw),
        ('MSP_ts',  test_probs_ts.max(axis=1),  preds_ts),
    ]
    for method in ['APS', 'RAPS']:
        alpha_use = 0.10
        ps = CONFORMAL_SETS.get(backbone,{}).get(method,{}).get(alpha_use)
        if ps is not None:
            conf_sc = np.array([1.0 / max(len(s), 1) for s in ps])
            score_cfgs.append((f'Conformal_{method}', conf_sc, preds_ts))

    for score_name, scores, preds in score_cfgs:
        covs, accs, risks, a100, a90, a80, arc = \
            selective_metrics(scores, preds, test_labels_)
        selective_rows.append({'backbone':backbone,'score':score_name,
                               'acc_at_100':a100,'acc_at_90':a90,
                               'acc_at_80':a80,'aurc':arc})
        SELECTIVE_RES[backbone][score_name] = {'coverages':covs,'accs':accs,'risks':risks}
        print(f'  {backbone[:18]:18s} | {score_name:22s} | '
              f'Acc@100={a100:.3f} @90={a90:.3f} @80={a80:.3f} AURC={arc:.4f}')

selective_df = pd.DataFrame(selective_rows)
selective_df.to_csv(
    f'{OUTPUT_DIR}/tables/table8_selective_classification.csv', index=False)
print('\nSelective classification table saved.')


## 15. Journal-Ready Figures

In [ ]:
# ── 15. Journal-Ready Figures ─────────────────────────────────────────────

n_bb = len(CFG.BACKBONES)

# 15a. Accuracy-Coverage and Risk-Coverage curves
fig, axes = plt.subplots(n_bb, 2, figsize=(12, 4*n_bb))
if n_bb == 1: axes = axes[np.newaxis, :]
for r, backbone in enumerate(CFG.BACKBONES):
    for score_name, data in SELECTIVE_RES[backbone].items():
        axes[r,0].plot(data['coverages'], data['accs'],  label=score_name)
        axes[r,1].plot(data['coverages'], data['risks'], label=score_name)
    axes[r,0].set_xlabel('Coverage'); axes[r,0].set_ylabel('Accuracy')
    axes[r,0].set_title(f'{backbone} — Accuracy-Coverage')
    axes[r,0].legend(fontsize=7)
    axes[r,1].set_xlabel('Coverage'); axes[r,1].set_ylabel('Risk')
    axes[r,1].set_title(f'{backbone} — Risk-Coverage')
    axes[r,1].legend(fontsize=7)
plt.tight_layout(); savefig('08_accuracy_risk_coverage_curves')
print('Saved: 08_accuracy_risk_coverage_curves')

# 15b. Conformal coverage summary
fig, axes = plt.subplots(1, len(CFG.ALPHAS), figsize=(6*len(CFG.ALPHAS), 5))
if len(CFG.ALPHAS) == 1: axes = [axes]
for ax, alpha in zip(axes, CFG.ALPHAS):
    sub = conformal_df[conformal_df['alpha'] == alpha]
    methods = sub['method'].unique()
    x = np.arange(n_bb); ww = 0.8 / len(methods)
    for i, method in enumerate(methods):
        vals = []
        for bb in CFG.BACKBONES:
            v = sub[sub['model']==bb]['empirical_coverage'].values
            vals.append(v[0] if len(v) else float('nan'))
        ax.bar(x + i*ww, vals, ww, label=method, alpha=0.8)
    ax.axhline(1-alpha, color='red', linestyle='--', label=f'Target {1-alpha:.0%}')
    ax.set_xticks(x + ww)
    ax.set_xticklabels([b.split('_')[0] for b in CFG.BACKBONES], rotation=15)
    ax.set_ylabel('Empirical Coverage'); ax.set_title(f'alpha={alpha}')
    ax.legend(fontsize=8); ax.set_ylim(0, 1.1)
plt.suptitle('Empirical vs Target Conformal Coverage', fontweight='bold')
plt.tight_layout(); savefig('09_conformal_coverage_summary')
print('Saved: 09_conformal_coverage_summary')

# 15c. Per-class set size distribution (RAPS, alpha=0.10)
for backbone in CFG.BACKBONES:
    ps = CONFORMAL_SETS.get(backbone,{}).get('RAPS',{}).get(0.10)
    if ps is None: continue
    tst_lbl = test_results[backbone]['labels']
    fig, axes = plt.subplots(1, CFG.NUM_CLASSES, figsize=(4*CFG.NUM_CLASSES, 4))
    for c in range(CFG.NUM_CLASSES):
        mask  = (tst_lbl == c)
        sizes = [len(s) for s, m in zip(ps, mask) if m]
        ax    = axes[c]
        if sizes:
            vals, cnts = np.unique(sizes, return_counts=True)
            ax.bar(vals, cnts / len(sizes), color=COLORS[c])
        ax.set_title(IDX_TO_CLASS[c], fontsize=9)
        ax.set_xlabel('Set size'); ax.set_ylabel('Fraction')
        ax.set_xticks(range(1, CFG.NUM_CLASSES + 1))
    plt.suptitle(f'{backbone} RAPS Set Size per Class (alpha=0.10)', fontweight='bold')
    plt.tight_layout()
    savefig(f'10_{backbone.replace("/","_")}_set_size_per_class')
    print(f'Saved: 10_{backbone}_set_size_per_class')

# 15d. Reliability diagrams all models
fig, axes = plt.subplots(n_bb, 2, figsize=(10, 4*n_bb))
if n_bb == 1: axes = axes[np.newaxis, :]
for r, backbone in enumerate(CFG.BACKBONES):
    tst_lbl = test_results[backbone]['labels']
    for col, (prbs, tag) in enumerate([
        (test_results[backbone]['probs'],    'Pre-TS'),
        (test_results[backbone]['probs_ts'], 'Post-TS'),
    ]):
        b_accs, b_confs, _ = reliability_diagram(prbs, tst_lbl)
        ax = axes[r, col]
        ax.bar(b_confs, b_accs, width=1/15, alpha=0.7)
        ax.plot([0,1],[0,1],'k--', lw=1)
        ece, _ = compute_ece(prbs, tst_lbl)
        ax.set_title(f'{backbone[:18]} {tag} ECE={ece:.3f}', fontsize=9)
        ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
plt.suptitle('Reliability Diagrams: All Models', fontweight='bold')
plt.tight_layout(); savefig('11_reliability_diagrams_all_models')
print('Saved: 11_reliability_diagrams_all_models')

# 15e. Per-class ambiguity bar chart
fig, axes = plt.subplots(1, n_bb, figsize=(6*n_bb, 4))
if n_bb == 1: axes = [axes]
for ax, backbone in zip(axes, CFG.BACKBONES):
    ps = CONFORMAL_SETS.get(backbone,{}).get('RAPS',{}).get(0.10)
    if ps is None: continue
    tst_lbl = test_results[backbone]['labels']
    cls_names, cls_mean_sz, cls_sing_r = [], [], []
    for c in range(CFG.NUM_CLASSES):
        mask   = (tst_lbl == c)
        sets_c = [s for s, m in zip(ps, mask) if m]
        if not sets_c: continue
        cls_names.append(IDX_TO_CLASS[c])
        cls_mean_sz.append(sum(len(s) for s in sets_c) / len(sets_c))
        cls_sing_r.append(sum(1 for s in sets_c if len(s)==1) / len(sets_c))
    x = np.arange(len(cls_names))
    ax.bar(x - 0.2, cls_mean_sz, 0.4, label='Mean set size', color='steelblue')
    ax2 = ax.twinx()
    ax2.bar(x + 0.2, cls_sing_r, 0.4, label='Singleton rate', color='salmon', alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(cls_names, rotation=15, ha='right', fontsize=8)
    ax.set_ylabel('Mean Set Size'); ax2.set_ylabel('Singleton Rate')
    ax.set_title(f'{backbone[:20]} Ambiguity')
    ax.legend(loc='upper left', fontsize=7); ax2.legend(loc='upper right', fontsize=7)
plt.suptitle('Per-Class Ambiguity (RAPS alpha=0.10)', fontweight='bold')
plt.tight_layout(); savefig('12_per_class_ambiguity')
print('Saved: 12_per_class_ambiguity')


## 16. Qualitative Panel: Confident Correct / Wrong / Abstained

In [ ]:
# ── 16. Qualitative Panel ─────────────────────────────────────────────────
def qualitative_panel(backbone, n_each=4):
    ts_probs = test_results[backbone]['probs_ts']
    tst_lbl  = test_results[backbone]['labels']
    preds_   = ts_probs.argmax(axis=1)
    confs    = ts_probs.max(axis=1)
    correct  = (preds_ == tst_lbl)
    thresh_abs = np.quantile(confs, 0.20)
    abstain    = confs < thresh_abs

    def get_subset(mask, n):
        idxs = np.where(mask)[0]
        if len(idxs) == 0: return []
        return np.random.choice(idxs, min(n, len(idxs)), replace=False).tolist()

    hi_conf = confs > np.quantile(confs, 0.80)
    cc_idxs = get_subset(correct & ~abstain & hi_conf, n_each)
    cw_idxs = get_subset(~correct & ~abstain & hi_conf, n_each)
    ab_idxs = get_subset(abstain, n_each)

    categories = [('Confident Correct', cc_idxs, 'green'),
                  ('Confident Wrong',   cw_idxs, 'red'),
                  ('Abstained',         ab_idxs, 'gray')]

    fig, axes_all = plt.subplots(3, n_each, figsize=(n_each*2.5, 8))
    for row_i, (cat_name, idxs, color) in enumerate(categories):
        for col_i in range(n_each):
            ax = axes_all[row_i][col_i]
            ax.axis('off')
            if col_i < len(idxs):
                idx = idxs[col_i]
                try:
                    img = Image.open(df_test.iloc[idx]['file_path']).convert('RGB')
                    ax.imshow(img)
                except Exception:
                    ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
                tc = IDX_TO_CLASS[tst_lbl[idx]]
                pc = IDX_TO_CLASS[preds_[idx]]
                ax.set_title(f'T:{tc[:4]} P:{pc[:4]}\n{confs[idx]:.2f}',
                             fontsize=7, color=color)
            if col_i == 0:
                ax.set_ylabel(cat_name, rotation=90, fontsize=9, color=color)
    plt.suptitle(f'Qualitative Panel — {backbone}', fontweight='bold')
    plt.tight_layout()
    savefig(f'13_{backbone.replace("/","_")}_qualitative_panel')
    print(f'Saved: 13_{backbone}_qualitative_panel')

for bb in CFG.BACKBONES:
    qualitative_panel(bb)


## 16b. Advanced Visualizations: GradCAM++, EigenCAM & Uncertainty

- **GradCAM++**: Gradient-weighted class activation map highlighting discriminative regions
- **EigenCAM**: PCA-based feature map visualization (works well on both CNN and ViT)
- **Uncertainty heatmap**: Test images sorted by entropy (most → least certain)
- **Uncertainty boundary**: Confidence vs entropy scatter, per-class entropy distribution


In [ ]:
# ── 16b. GradCAM++ / EigenCAM / Uncertainty Visualizations ──────────────────
try:
    from pytorch_grad_cam import GradCAMPlusPlus, EigenCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    _cam_ok = True
except ImportError:
    import subprocess as _sp
    _sp.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'grad-cam'])
    from pytorch_grad_cam import GradCAMPlusPlus, EigenCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    _cam_ok = True


def _get_cam_layers(model, backbone_name):
    '''Return (target_layers, reshape_transform) for pytorch-grad-cam.'''
    if 'convnext' in backbone_name.lower():
        # Last ConvNeXt stage — plain CNN feature maps
        return [model.stages[-1]], None
    elif 'vit' in backbone_name.lower():
        # ViT patch tokens need reshaping: [B, N+1, C] -> [B, C, H, W]
        def _vit_reshape(tensor, h=14, w=14):
            result = tensor[:, 1:].reshape(tensor.size(0), h, w, tensor.size(2))
            return result.permute(0, 3, 1, 2)
        return [model.blocks[-1].norm1], _vit_reshape
    else:
        children = list(model.children())
        return [children[max(-2, -len(children))]], None


def plot_cam_panel(backbone_name, model, df_test_df, probs_ts, labels_arr,
                   device, n_per_class=3):
    model.eval()
    target_layers, reshape_fn = _get_cam_layers(model, backbone_name)
    preds = probs_ts.argmax(axis=1)

    rows = []  # (img_np, gradcam_overlay, eigen_overlay, label, conf)
    for c in range(CFG.NUM_CLASSES):
        # prefer correctly classified; fall back to any sample of class c
        mask_ok  = (labels_arr == c) & (preds == c)
        mask_any = (labels_arr == c)
        idxs = np.where(mask_ok)[0] if mask_ok.any() else np.where(mask_any)[0]
        if len(idxs) == 0:
            continue
        chosen = idxs[:n_per_class]
        for idx in chosen:
            fp = df_test_df.iloc[idx]['file_path']
            try:
                img_pil = Image.open(fp).convert('RGB').resize((224, 224))
            except Exception:
                continue
            img_np      = np.array(img_pil, dtype=np.float32) / 255.0
            input_t     = eval_transform(img_pil).unsqueeze(0).to(device)
            cam_target  = [ClassifierOutputTarget(int(c))]

            try:
                cam_pp = GradCAMPlusPlus(model=model, target_layers=target_layers,
                                         reshape_transform=reshape_fn)
                pp_mask    = cam_pp(input_tensor=input_t, targets=cam_target)
                pp_overlay = show_cam_on_image(img_np, pp_mask[0], use_rgb=True)
            except Exception as e:
                print(f'  GradCAM++ skipped ({e})')
                pp_overlay = (img_np * 255).astype(np.uint8)

            try:
                ecam       = EigenCAM(model=model, target_layers=target_layers,
                                      reshape_transform=reshape_fn)
                e_mask     = ecam(input_tensor=input_t, targets=cam_target)
                e_overlay  = show_cam_on_image(img_np, e_mask[0], use_rgb=True)
            except Exception as e:
                print(f'  EigenCAM skipped ({e})')
                e_overlay = (img_np * 255).astype(np.uint8)

            rows.append((img_np, pp_overlay, e_overlay,
                         IDX_TO_CLASS[c], float(probs_ts[idx].max())))

    if not rows:
        print(f'  No images available for CAM panel — skipping.')
        return

    n_r = len(rows)
    fig, axes = plt.subplots(n_r, 3, figsize=(10, n_r * 2.6 + 0.5))
    if n_r == 1: axes = axes[np.newaxis, :]
    axes[0, 0].set_title('Original',   fontsize=9, fontweight='bold')
    axes[0, 1].set_title('GradCAM++',  fontsize=9, fontweight='bold')
    axes[0, 2].set_title('EigenCAM',   fontsize=9, fontweight='bold')
    for r, (orig, pp, ec, cls_name, conf) in enumerate(rows):
        axes[r,0].imshow(orig); axes[r,0].axis('off')
        axes[r,0].set_ylabel(f'{cls_name}\nConf={conf:.2f}', fontsize=7)
        axes[r,1].imshow(pp);   axes[r,1].axis('off')
        axes[r,2].imshow(ec);   axes[r,2].axis('off')
    plt.suptitle(f'{backbone_name} — GradCAM++ & EigenCAM', fontweight='bold')
    plt.tight_layout()
    savefig(f'14_{backbone_name.replace("/","_")}_gradcam_eigencam')
    print(f'Saved: 14_{backbone_name}_gradcam_eigencam')


def plot_uncertainty_analysis(backbone_name, probs_ts, labels_arr, df_test_df, output_dir):
    preds      = probs_ts.argmax(axis=1)
    entropy_   = -(probs_ts * np.log(probs_ts + 1e-10)).sum(axis=1)
    confidence = probs_ts.max(axis=1)
    correct    = (preds == labels_arr)

    # ── Fig A: Entropy violin per class + conf vs acc bar ──────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Violin: uncertainty distribution per class
    ent_per_cls = [entropy_[labels_arr == c] for c in range(CFG.NUM_CLASSES)]
    ent_per_cls = [e if len(e) > 0 else np.array([0.0]) for e in ent_per_cls]
    vp = axes[0].violinplot(ent_per_cls, positions=range(CFG.NUM_CLASSES),
                             showmedians=True, showextrema=True)
    for i, body in enumerate(vp['bodies']):
        body.set_facecolor(COLORS[i % len(COLORS)]); body.set_alpha(0.7)
    axes[0].set_xticks(range(CFG.NUM_CLASSES))
    axes[0].set_xticklabels([IDX_TO_CLASS[c] for c in range(CFG.NUM_CLASSES)],
                             rotation=15, ha='right', fontsize=8)
    axes[0].set_ylabel('Entropy H(p)'); axes[0].set_title('Uncertainty per Class')

    # Scatter: uncertainty boundary (entropy vs confidence)
    c_col = np.where(correct, 1, 0)
    sc = axes[1].scatter(entropy_, confidence, c=c_col, cmap='RdYlGn',
                          vmin=0, vmax=1, alpha=0.35, s=6)
    cbar = plt.colorbar(sc, ax=axes[1])
    cbar.set_ticks([0.1, 0.9]); cbar.set_ticklabels(['Wrong','Correct'])
    axes[1].set_xlabel('Entropy (uncertainty)'); axes[1].set_ylabel('Max confidence')
    axes[1].set_title('Uncertainty Boundary\n(correct=green, wrong=red)')

    # Bar: mean confidence vs accuracy per class
    conf_per = [confidence[labels_arr == c].mean() if (labels_arr==c).any() else 0
                for c in range(CFG.NUM_CLASSES)]
    acc_per  = [correct[labels_arr == c].mean()    if (labels_arr==c).any() else 0
                for c in range(CFG.NUM_CLASSES)]
    x = np.arange(CFG.NUM_CLASSES)
    axes[2].bar(x-0.2, conf_per, 0.38, label='Mean confidence', color='steelblue')
    axes[2].bar(x+0.2, acc_per,  0.38, label='Accuracy',        color='salmon')
    axes[2].plot([-.5, CFG.NUM_CLASSES-.5],[1,1],'k--',lw=0.7)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels([IDX_TO_CLASS[c] for c in range(CFG.NUM_CLASSES)],
                             rotation=15, ha='right', fontsize=8)
    axes[2].set_title('Confidence vs Accuracy per Class'); axes[2].legend(fontsize=8)
    axes[2].set_ylim(0, 1.15)

    plt.suptitle(f'{backbone_name} — Uncertainty Analysis', fontweight='bold')
    plt.tight_layout()
    savefig(f'15_{backbone_name.replace("/","_")}_uncertainty_analysis')
    print(f'Saved: 15_{backbone_name}_uncertainty_analysis')

    # ── Fig B: Uncertainty Heatmap (image grid sorted by entropy) ─────────────
    n_show = min(40, len(entropy_))
    n_half = n_show // 2
    order  = np.argsort(entropy_)
    # show n_half most certain + n_half least certain
    selected = np.concatenate([order[:n_half], order[-n_half:]])
    n_cols   = min(8, n_show); n_rows = (len(selected) + n_cols - 1) // n_cols

    fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.8, n_rows * 1.8))
    axes2 = np.array(axes2).reshape(-1) if n_rows > 1 else np.array([axes2]).reshape(-1)

    for i, idx in enumerate(selected):
        ax  = axes2[i]
        fp  = df_test_df.iloc[idx]['file_path']
        try:
            img = Image.open(fp).convert('RGB').resize((80, 80))
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', fontsize=6)
        ec = 'limegreen' if correct[idx] else 'crimson'
        for sp in ax.spines.values(): sp.set_edgecolor(ec); sp.set_linewidth(3)
        ax.set_title(f'H={entropy_[idx]:.2f}', fontsize=5, pad=1)
        ax.axis('off')
    for j in range(len(selected), len(axes2)): axes2[j].axis('off')

    # Divider line between most-certain and least-certain halves
    fig2.text(0.5, 0.5,
              '<-- Most certain            Least certain -->',
              ha='center', va='center', fontsize=8, color='gray',
              transform=fig2.transFigure)
    plt.suptitle(
        f'{backbone_name} — Uncertainty Heatmap\n'
        'Sorted by entropy H(p) | green border=correct, red=wrong',
        fontsize=9, fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.93])
    savefig(f'16_{backbone_name.replace("/","_")}_uncertainty_heatmap')
    print(f'Saved: 16_{backbone_name}_uncertainty_heatmap')


# Run for all trained models
for bb in CFG.BACKBONES:
    model    = trained_models[bb]
    probs_ts = test_results[bb]['probs_ts']
    lbl_arr  = test_results[bb]['labels']
    if _cam_ok:
        plot_cam_panel(bb, model, df_test, probs_ts, lbl_arr, DEVICE, n_per_class=3)
    plot_uncertainty_analysis(bb, probs_ts, lbl_arr, df_test, OUTPUT_DIR)


## 17. Journal Result Tables

In [ ]:
# ── 17. Result Tables ─────────────────────────────────────────────────────

# Table 5: Classification performance
t5_rows = []
for backbone, res in test_results.items():
    labels_ = res['labels']
    preds_  = res['probs'].argmax(axis=1)
    per_cls = f1_score(labels_, preds_, average=None, zero_division=0)
    row = {'Model':backbone,'Accuracy':f"{res['accuracy']:.4f}",
           'Macro_F1':f"{res['macro_f1']:.4f}",'AUROC':f"{res['auroc']:.4f}"}
    for c in range(CFG.NUM_CLASSES):
        cn = IDX_TO_CLASS[c]
        row[f'{cn}_F1'] = f'{per_cls[c]:.4f}' if c < len(per_cls) else 'N/A'
    t5_rows.append(row)
t5 = pd.DataFrame(t5_rows)
t5.to_csv(f'{OUTPUT_DIR}/tables/table5_classification.csv', index=False)
print('Table 5 — Classification Performance:')
print(t5.to_string(index=False))

# Table 6: Calibration
t6_rows = []
for backbone, cr in cal_results.items():
    t6_rows.append({'Model':backbone,'ECE_pre':f"{cr['ece_pre']:.4f}",
                    'ECE_post':f"{cr['ece_post']:.4f}",'MCE_post':f"{cr['mce_post']:.4f}",
                    'T_star':f"{cr['T']:.4f}"})
t6 = pd.DataFrame(t6_rows)
t6.to_csv(f'{OUTPUT_DIR}/tables/table6_calibration.csv', index=False)
print('\nTable 6 — Calibration:')
print(t6.to_string(index=False))

# Table 7: already saved — print summary
show_cols = ['model','method','alpha','target_coverage','empirical_coverage',
             'avg_set_size','singleton_rate']
print('\nTable 7 — Conformal Coverage (first rows):')
print(conformal_df[show_cols].head(12).to_string(index=False))

# Table 8: already saved — print summary
print('\nTable 8 — Selective Classification:')
print(selective_df.to_string(index=False))

# Table 9: per-class ambiguity
t9_rows = []
for backbone in CFG.BACKBONES:
    ps = CONFORMAL_SETS.get(backbone,{}).get('RAPS',{}).get(0.10)
    if ps is None: continue
    tst_lbl = test_results[backbone]['labels']
    for c in range(CFG.NUM_CLASSES):
        mask   = (tst_lbl == c)
        sets_c = [s for s, m in zip(ps, mask) if m]
        if not sets_c: continue
        t9_rows.append({
            'Model':backbone,'Class':IDX_TO_CLASS[c],
            'Mean_set_size': f'{sum(len(s) for s in sets_c)/len(sets_c):.3f}',
            'Singleton_rate':f'{sum(1 for s in sets_c if len(s)==1)/len(sets_c):.3f}'
        })
t9 = pd.DataFrame(t9_rows)
t9.to_csv(f'{OUTPUT_DIR}/tables/table9_per_class_ambiguity.csv', index=False)
print('\nTable 9 — Per-Class Ambiguity:')
print(t9.to_string(index=False))


## 18. Optional Ensemble (if ≥ 2 models trained)

In [ ]:
# ── 18. Ensemble ──────────────────────────────────────────────────────────
if len(trained_models) >= 2:
    print('Building ensemble from:', list(trained_models.keys()))

    ensemble_probs = np.mean(
        [test_results[bb]['probs_ts'] for bb in CFG.BACKBONES], axis=0)
    ens_labels = test_results[CFG.BACKBONES[0]]['labels']
    ens_preds  = ensemble_probs.argmax(axis=1)

    ens_acc    = accuracy_score(ens_labels, ens_preds)
    ens_mac_f1 = f1_score(ens_labels, ens_preds, average='macro', zero_division=0)
    try:
        ens_auroc = roc_auc_score(ens_labels, ensemble_probs,
                                   multi_class='ovr', average='macro')
    except Exception:
        ens_auroc = float('nan')

    print(f'  Ensemble Accuracy   : {ens_acc:.4f}')
    print(f'  Ensemble Macro-F1   : {ens_mac_f1:.4f}')
    print(f'  Ensemble AUROC(OVR) : {ens_auroc:.4f}')

    # Ensemble conformal (RAPS, alpha=0.10)
    ens_cal_probs = np.mean(
        [torch.softmax(
             torch.tensor(cal_results[bb]['cal_logits']) / cal_results[bb]['T'],
             dim=-1).numpy()
         for bb in CFG.BACKBONES], axis=0)
    ens_cal_lbl = cal_results[CFG.BACKBONES[0]]['cal_labels']
    alpha_ens   = 0.10
    cal_s_ens   = raps_scores(ens_cal_probs, ens_cal_lbl)
    qhat_ens    = conformal_quantile(cal_s_ens, alpha_ens)
    ps_ens      = predict_sets_raps(ensemble_probs, qhat_ens)
    cov_ens, sz_ens, sing_ens, *_ = eval_conformal(ps_ens, ens_labels, CFG.NUM_CLASSES)
    print(f'  Ensemble RAPS alpha=0.10: coverage={cov_ens:.3f} sz={sz_ens:.2f}')

    np.savez(f'{OUTPUT_DIR}/predictions/ensemble_test.npz',
             probs=ensemble_probs, labels=ens_labels)
    pd.DataFrame([{'Model':'Ensemble','Accuracy':f'{ens_acc:.4f}',
                   'Macro_F1':f'{ens_mac_f1:.4f}','AUROC':f'{ens_auroc:.4f}'}]).to_csv(
        f'{OUTPUT_DIR}/tables/table5_ensemble.csv', index=False)
else:
    print('Only one model trained — skipping ensemble.')


## 19. Output Packaging

In [ ]:
# ── 19. Output Packaging ──────────────────────────────────────────────────
import zipfile, pathlib

output_root = pathlib.Path(OUTPUT_DIR)
zip_path    = '/kaggle/working/gi_conformal_outputs.zip'

print('Files in output directory:')
all_files = sorted(output_root.rglob('*'))
total_kb  = 0
for f in all_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        total_kb += size_kb
        print(f'  {str(f.relative_to(output_root)):60s}  {size_kb:8.1f} KB')
print(f'\nTotal size: {total_kb/1024:.1f} MB')

print(f'\nCreating zip: {zip_path}')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in all_files:
        if f.is_file():
            zf.write(f, f.relative_to(output_root.parent))

zip_size = pathlib.Path(zip_path).stat().st_size / 1e6
print(f'Zip size: {zip_size:.1f} MB')
print()
print('=' * 60)
print('DONE! Download gi_conformal_outputs.zip from the Kaggle Output panel.')
print('=' * 60)
